# 02_Chunk — Segmentation sémantique

**Objectif métier** : produire des chunks courts et thématiquement homogènes
(un chunk = une facette d'une entité : localisation, conditions, documents...)
plutôt que des blocs de texte monolithiques — c'est ce qui permet au
retrieval hybride de la Phase 5 de remonter précisément le bon fragment
(ex: une question sur les documents requis doit matcher le chunk
"documents", pas noyer cette info dans un paragraphe de description).

**Cibles du guide** : Service -> 4 chunks min, Centre -> 3 chunks,
Programme -> 3 chunks, FAQ -> 1-2 chunks. Chunks conditions/documents
priorisés (ce sont les questions réelles des usagers). Métadonnées
obligatoires sur chaque chunk : `entity_type`, `entity_id`, `chunk_type`,
`population_cible`, `institutions`, `langue`.

In [1]:
import sys, re
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from etl_lib.ontology import INSTITUTION_MAP, POPULATION_CIBLES, clean_text
from etl_lib.io_utils import load_jsonl, save_jsonl

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
CHUNKS_DIR = Path.cwd().parent / "data" / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

services = load_jsonl(PROCESSED_DIR / "services_unified.jsonl")
centres = load_jsonl(PROCESSED_DIR / "centres_normalized.jsonl")
programmes = load_jsonl(PROCESSED_DIR / "programmes_2027.jsonl")
faqs = load_jsonl(PROCESSED_DIR / "faq_aos.jsonl")
print(f"services={len(services)} centres={len(centres)} programmes={len(programmes)} faq={len(faqs)}")

_AR_RE = re.compile(r"[\u0600-\u06FF]")


def detect_langue(text: str) -> str:
    if not text:
        return "ar"
    ar = len(_AR_RE.findall(text))
    return "ar" if ar / max(len(text.replace(' ', '')), 1) > 0.25 else "fr"


def make_chunk(entity_type, entity_id, chunk_type, text, **meta):
    text = clean_text(text)
    if not text:
        return None
    return {
        "chunk_id": f"{entity_id}_{chunk_type}",
        "text_bilingual": text,
        "type": entity_type.lower(),
        "entity_type": entity_type,
        "entity_id": entity_id,
        "chunk_type": chunk_type,
        "langue": detect_langue(text),
        **meta,
    }


services=61 centres=3328 programmes=7 faq=8


## 1. Chunks Service (4 par service : description, conditions, documents, institutions)

In [2]:
service_chunks = []
for s in services:
    inst_labels = [f"{c} ({INSTITUTION_MAP.get(c, {}).get('fr', c)})" for c in s["institutions"]]
    meta = {
        "population_cible": s["categorie"],
        "institutions": s["institutions"],
        "axe_programme": s.get("axe_programme_ar", ""),
    }

    chunks = [
        make_chunk("Service", s["id"], "description",
                   f"{s['service_ar']}. {s['description_ar']}", **meta),
        make_chunk("Service", s["id"], "conditions",
                   "Conditions: " + " | ".join(s["conditions_ar"]), **meta),
        make_chunk("Service", s["id"], "documents",
                   "Documents requis: " + " | ".join(s["documents_ar"]), **meta),
        make_chunk("Service", s["id"], "institutions",
                   f"Institutions: {', '.join(inst_labels)}. Axe: {s.get('axe_programme_ar', '')}"
                   + (" (dernier recours, aucune institution specifique detectee)" if s["eps_fallback"] else ""),
                   **meta),
    ]
    service_chunks.extend(c for c in chunks if c)

print(f"Chunks Service : {len(service_chunks)} ({len(service_chunks)/len(services):.1f}/service, cible >= 4 quand conditions+documents presents)")


Chunks Service : 244 (4.0/service, cible >= 4 quand conditions+documents presents)


## 2. Chunks Centre (3 par centre : localisation, activité, programme)

In [3]:
centre_chunks = []
for c in centres:
    meta = {
        "population_cible": c.get("personnes_cibles", ""),
        "institutions": c["institutions"],
        "region": c["region"], "delegation": c["delegation"], "commune": c["commune"],
    }
    loc_text = f"{c['nom']}. {c['adresse']}. Commune: {c['commune']}, Delegation: {c['delegation']}, Region: {c['region']}."
    act_text = (f"Activite: {c['activite']}. {c['description']}. "
                f"Capacite: {c['capacite'] or 'inconnue'}. Milieu: {c['milieu']}. Propriete: {c['propriete']}.")
    prog_text = f"Programme: {c['programme']}. Axe: {c['axe']}. Population ciblee: {c['personnes_cibles']}."

    chunks = [
        make_chunk("Centre", c["id"], "localisation", loc_text, **meta),
        make_chunk("Centre", c["id"], "activite", act_text, **meta),
        make_chunk("Centre", c["id"], "programme", prog_text, **meta),
    ]
    centre_chunks.extend(x for x in chunks if x)

print(f"Chunks Centre : {len(centre_chunks)} ({len(centre_chunks)/len(centres):.1f}/centre, cible = 3)")


Chunks Centre : 9984 (3.0/centre, cible = 3)


## 3. Chunks Programme 2027 (3 par programme)

In [4]:
programme_chunks = []
for p in programmes:
    meta = {
        "population_cible": p.get("population_cible", ""),
        "institutions": p.get("institutions_liees", []),
        "budget": p.get("budget"),
    }
    desc_text = f"{p['titre_ar']} / {p['titre_fr']}. {p['description_ar']}"
    obj_text = f"Objectifs: {p['objectifs_ar']}. Indicateurs: {' | '.join(p.get('indicateurs', []))}. Budget: {p.get('budget', 0):,.0f} MAD."
    inst_text = f"Institutions liees: {', '.join(p.get('institutions_liees', []))}. Population cible: {p.get('population_cible', '')}."

    chunks = [
        make_chunk("Programme", p["id"], "description", desc_text, **meta),
        make_chunk("Programme", p["id"], "indicateurs", obj_text, **meta),
        make_chunk("Programme", p["id"], "institutions", inst_text, **meta),
    ]
    programme_chunks.extend(x for x in chunks if x)

print(f"Chunks Programme : {len(programme_chunks)} ({len(programme_chunks)/max(len(programmes),1):.1f}/programme, cible = 3)")


Chunks Programme : 21 (3.0/programme, cible = 3)


## 4. Chunks FAQ (1 par question — déjà atomique)

In [5]:
faq_chunks = []
for f in faqs:
    meta = {"population_cible": "", "institutions": [], "section": f.get("section_ar", "")}
    text = f"{f['question_ar']} {f['reponse_ar']}"
    c = make_chunk("FAQ", f["id"], "qa", text, **meta)
    if c:
        faq_chunks.append(c)

print(f"Chunks FAQ : {len(faq_chunks)} ({len(faq_chunks)/max(len(faqs),1):.1f}/FAQ, cible 1-2)")


Chunks FAQ : 8 (1.0/FAQ, cible 1-2)


In [6]:
# === FAQ_EXTRA_INGESTION_BLOCK ===
# Chunks pour les 121 enregistrements issus de data_faq (voir
# scripts/normalize_faq_dataset.py) : familles A (service) + C (generique)
# -> 2 chunks chacune (question seule / question+reponse, pour permettre le
# matching question-a-question ET question-a-contenu, cf. discussion metier
# du 2026-08-12) ; famille B (fiches Institution) -> 1 chunk profil, rattache
# a l'entite Institution existante (entity_id = code, ex "CEF").
faq_extra_service = load_jsonl(PROCESSED_DIR / "faq_extra_service.jsonl")
faq_extra_general = load_jsonl(PROCESSED_DIR / "faq_extra_general.jsonl")
institution_enrich = load_jsonl(PROCESSED_DIR / "institution_enrich.jsonl")

faq_extra_chunks = []
for f in faq_extra_service + faq_extra_general:
    meta = {
        "population_cible": " | ".join(f.get("population_cible", [])),
        "institutions": f.get("institutions", []),
        "section": f.get("category", ""),
    }
    q = make_chunk("FAQ", f["id"], "faq_question", f["question_ar"], **meta)
    c = make_chunk("FAQ", f["id"], "faq_content", f"{f['question_ar']} {f['reponse_ar']}", **meta)
    faq_extra_chunks.extend(x for x in (q, c) if x)

institution_profile_chunks = []
for ip in institution_enrich:
    meta = {"population_cible": " | ".join(ip.get("population_cible", [])), "institutions": [ip["institution_code"]]}
    c = make_chunk("Institution", ip["institution_code"], "institution_profile",
                    f"{ip['question_ar']} {ip['reponse_ar']}", **meta)
    if c:
        institution_profile_chunks.append(c)

print(f"Chunks FAQ (data_faq)        : {len(faq_extra_chunks)} ({len(faq_extra_service) + len(faq_extra_general)} entrees x2)")
print(f"Chunks Institution (profils) : {len(institution_profile_chunks)} ({len(institution_enrich)} codes)")


Chunks FAQ (data_faq)        : 226 (113 entrees x2)
Chunks Institution (profils) : 8 (8 codes)


## 5. Sauvegarde + rapport

In [7]:
# === FAQ_EXTRA_INGESTION_BLOCK ===
all_chunks = (
    service_chunks + centre_chunks + programme_chunks + faq_chunks
    + faq_extra_chunks + institution_profile_chunks
)
save_jsonl(CHUNKS_DIR / "chunks_all.jsonl", all_chunks)

from collections import Counter
by_type = Counter(c["entity_type"] for c in all_chunks)
by_lang = Counter(c["langue"] for c in all_chunks)

print(f"Total chunks : {len(all_chunks)}")
print(f"Par entite   : {dict(by_type)}")
print(f"Par langue   : {dict(by_lang)}")
print(f"\nCible guide (8000-15000 chunks pour un jeu de donnees complet) : "
      f"{'dans la fourchette' if 3000 <= len(all_chunks) else 'en dessous (jeu de donnees plus reduit que prevu par le guide, coherent avec les volumes reels de rag data/)'}")
print("\n\u2705 02_Chunk termine.")


Total chunks : 10491
Par entite   : {'Service': 244, 'Centre': 9984, 'Programme': 21, 'FAQ': 234, 'Institution': 8}
Par langue   : {'ar': 591, 'fr': 9900}

Cible guide (8000-15000 chunks pour un jeu de donnees complet) : dans la fourchette

✅ 02_Chunk termine.
